# Data Prep & Transformation | Aula 1 · Prática
## Diagnóstico das fontes transacionais da Olist

**Prof. Luciano Gillieron Gavinho, PhD.** · MBA em Engenharia de Dados · Universidade Presbiteriana Mackenzie

A Olist, um marketplace brasileiro de comércio eletrônico, publicou no GitHub (site público de hospedagem de código) uma amostra do seu banco de dados. Segundo o próprio repositório, as tabelas foram extraídas **diretamente do ambiente transacional, sem qualquer tratamento**. Ambiente transacional, ou **OLTP** (*Online Transaction Processing*, processamento de transações em tempo real), é o sistema que executa a operação do dia a dia: registra pedidos, pagamentos e entregas. É exatamente o ponto de partida desta disciplina: dado transacional bruto.

**Seu papel hoje:** você é o(a) engenheiro(a) de dados que acabou de receber esses arquivos. Antes de construir qualquer **pipeline** (sequência automatizada de etapas que leva o dado da fonte até o uso), precisa responder: *o que eu tenho em mãos, qual o grão de cada tabela, e o que vai quebrar se eu usar isso direto?*

**Como trabalhar**
- Em grupos. Cada pessoa roda o próprio notebook, e o grupo consolida os achados.
- Use **SQL (DuckDB)** como ferramenta principal; pandas quando fizer sentido.
- Comece pela **seção A · Aquecimento**: exemplos resolvidos que mostram a sintaxe que você vai usar.
- Anote cada problema encontrado na **Tabela de Achados**, no fim do notebook. Ela é o produto da prática.

### Termos usados neste notebook

| Termo | O que significa |
|---|---|
| **Colab** | Google Colaboratory: ambiente gratuito do Google que roda notebooks Python no navegador, sem instalar nada. |
| **Notebook** | Documento que mistura texto e células de código executáveis. Execute uma célula com Shift + Enter. |
| **CSV** | *Comma-Separated Values*: arquivo de texto em que cada linha é um registro e as colunas são separadas por vírgula. |
| **SQL** | *Structured Query Language*: linguagem padrão para consultar e transformar dados em tabelas. |
| **DuckDB** | Banco de dados analítico que roda dentro do próprio notebook e executa SQL direto sobre arquivos CSV. |
| **pandas** | Biblioteca Python para manipular tabelas em memória. |
| **DataFrame** | A tabela do pandas: linhas e colunas com nome e tipo. |
| **View** | Consulta salva com um nome. Aqui, cada view aponta para um arquivo CSV, que é lido a cada consulta. |
| **Registro** | Uma linha lógica de uma tabela. Nem sempre equivale a uma linha física do arquivo (você vai ver por quê). |
| **Grão** | O que exatamente uma linha da tabela representa: um pedido, um item de pedido, um pagamento. |
| **Chave** | Coluna, ou conjunto de colunas, cujo valor identifica uma única linha. **Chave composta** é a formada por mais de uma coluna. |
| **NULL** | Ausência de valor. Não é zero nem texto vazio: é "não informado". |
| **Cardinalidade** | Número de valores distintos de uma coluna. |
| **Tipo de dado** | Como a coluna é armazenada. `VARCHAR`: texto. `BIGINT` e `int64`: número inteiro. `DOUBLE`: número decimal. `TIMESTAMP`: data e hora. |
| **Inferência de tipo** | Decisão automática da ferramenta sobre o tipo de cada coluna ao ler um arquivo. |
| **Integridade referencial** | Garantia de que toda referência a outra tabela aponta para um registro que existe (todo item aponta para um pedido real, por exemplo). |
| **CEP** | Código de Endereçamento Postal. Os arquivos da Olist trazem só o prefixo de 5 dígitos. |

## 0 · Preparação do ambiente
Execute as duas células abaixo. A primeira faz um `git clone`, isto é, baixa uma cópia do repositório oficial da Olist (cerca de 125 MB), e instala o DuckDB com `pip`, o instalador de pacotes Python. A segunda abre uma conexão DuckDB e registra cada arquivo CSV como uma *view* com nome curto, além de criar a função `sql()`, que executa uma consulta e devolve o resultado como DataFrame.

In [20]:
import os, subprocess
if not os.path.exists("work-at-olist-data"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/olist/work-at-olist-data.git"], check=True)
subprocess.run(["pip", "install", "-q", "duckdb"], check=False)
print(sorted(os.listdir("work-at-olist-data/datasets")))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [21]:
import duckdb, pandas as pd
DATA = "work-at-olist-data/datasets"
con = duckdb.connect()

FONTES = {
    "customers":  "olist_customers_dataset.csv",
    "geolocation":"olist_geolocation_dataset.csv",
    "orders":     "olist_orders_dataset.csv",
    "items":      "olist_order_items_dataset.csv",
    "payments":   "olist_order_payments_dataset.csv",
    "reviews":    "olist_order_reviews_dataset.csv",
    "products":   "olist_products_dataset.csv",
    "sellers":    "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
for nome, arq in FONTES.items():
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_csv_auto('{DATA}/{arq}', header=true)")

def sql(q):
    """Executa uma consulta no DuckDB e devolve um DataFrame."""
    return con.sql(q).df()

sql("SELECT * FROM orders LIMIT 3")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


## A · Aquecimento: exemplos resolvidos
Antes dos exercícios, cinco consultas prontas sobre as tabelas `orders` (pedidos) e `payments` (pagamentos). Execute, leia o resultado e observe a sintaxe. Elas **não** respondem aos exercícios; mostram as ferramentas que você vai combinar.

### A.1 · Que colunas existem e de que tipo?
`DESCRIBE` lista as colunas de uma tabela com o tipo que o DuckDB inferiu ao ler o CSV.

In [22]:
sql("DESCRIBE orders")

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


### A.2 · Quantas linhas e como elas são?
`COUNT(*)` conta todas as linhas. `LIMIT n` devolve só as n primeiras, para olhar o dado sem carregar tudo. Repare que são duas consultas: a primeira é mostrada com `display()`, a segunda é o resultado final da célula.

In [23]:
display(sql("SELECT COUNT(*) AS total_de_pedidos FROM orders"))
sql("SELECT order_id, order_status, order_purchase_timestamp FROM orders LIMIT 5")

,total_de_pedidos
0,99441


,order_id,order_status,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39


### A.3 · Quantos valores faltam em cada coluna?
`COUNT(coluna)` conta só as linhas em que a coluna **não** é NULL. Logo, `COUNT(*) - COUNT(coluna)` é o número de nulos daquela coluna.

In [24]:
sql("""
SELECT
  COUNT(*)                                       AS linhas,
  COUNT(*) - COUNT(order_approved_at)            AS nulos_aprovacao,
  COUNT(*) - COUNT(order_delivered_carrier_date) AS nulos_entrega_transportadora,
  COUNT(*) - COUNT(order_delivered_customer_date) AS nulos_entrega_cliente
FROM orders
""")

,linhas,nulos_aprovacao,nulos_entrega_transportadora,nulos_entrega_cliente
0,99441,160,1783,2965


### A.4 · Perfil completo em uma linha de código
`SUMMARIZE` é um comando do DuckDB que faz o **perfilamento** (levantamento sistemático de tipo, mínimo, máximo, valores distintos aproximados e percentual de nulos) de todas as colunas de uma vez. É o primeiro comando a rodar quando você recebe uma tabela nova.

**Atenção:** a coluna `approx_unique` é uma **estimativa** de valores distintos, calculada por um algoritmo aproximado para ser rápida. Compare a estimativa de `order_id` com o valor exato, `COUNT(DISTINCT order_id)`, na segunda consulta. Para testar se uma chave é única, use sempre a contagem exata.

In [25]:
display(sql("SUMMARIZE payments"))
sql("SELECT COUNT(DISTINCT order_id) AS order_id_distintos_exato FROM payments")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,87618,None,None,None,None,None,103886,0.0
1,payment_sequential,BIGINT,1,29,27,1.0926785129853878,0.7065837791949948,1,1,1,103886,0.0
2,payment_type,VARCHAR,boleto,voucher,5,None,None,None,None,None,103886,0.0
3,payment_installments,BIGINT,0,24,25,2.853608763452246,2.6873066442601528,1,1,4,103886,0.0
4,payment_value,DOUBLE,0.0,13664.08,31557,154.10038041698365,217.49406386472384,56.77380184130726,100.10889896724083,171.75934799732582,103886,0.0


,order_id_distintos_exato
0,99440


### A.5 · Contar por categoria
`GROUP BY` agrupa as linhas que têm o mesmo valor numa coluna; as funções de agregação (`COUNT`, `SUM`, `AVG`) são calculadas por grupo. `ORDER BY ... DESC` ordena do maior para o menor. A segunda parte mostra a mesma contagem feita em pandas, para comparar as duas sintaxes.

In [26]:
display(sql("""
SELECT payment_type,
       COUNT(*)                     AS pagamentos,
       ROUND(AVG(payment_value), 2) AS valor_medio
FROM payments
GROUP BY payment_type
ORDER BY pagamentos DESC
"""))

pag = sql("SELECT * FROM payments")          # traz a tabela inteira para um DataFrame
pag["payment_type"].value_counts()           # mesma contagem, agora em pandas

,payment_type,pagamentos,valor_medio
0,credit_card,76795,163.32
1,boleto,19784,145.03
2,voucher,5775,65.70
3,debit_card,1529,142.57
4,not_defined,3,0.00


,count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


> **Resumo das ferramentas:** `DESCRIBE` (tipos), `COUNT(*)` e `COUNT(coluna)` (linhas e nulos), `COUNT(DISTINCT coluna)` (valores distintos), `SUMMARIZE` (perfil), `GROUP BY` (contar por grupo). Com elas você resolve a maior parte dos exercícios. As dicas de cada exercício explicam o que mais for preciso.

## 1 · Inventário: quantos registros existem de fato?
Para cada uma das 9 fontes, conte **(a)** o número de registros que o DuckDB lê e **(b)** o número de linhas físicas do arquivo (desconte o cabeçalho).

**Pergunte-se:** os dois números batem em todas as tabelas? Se não batem, por quê?

In [10]:
linhas = []
for nome, arq in FONTES.items():
    registros = con.sql(f"SELECT COUNT(*) FROM {nome}").fetchone()[0]
    fisicas = sum(1 for _ in open(f"{DATA}/{arq}", encoding="utf-8")) - 1
    linhas.append((nome, registros, fisicas, fisicas - registros))
inv = pd.DataFrame(linhas, columns=["fonte", "registros", "linhas_no_arquivo", "diferenca"])
inv

,fonte,registros,linhas_no_arquivo,diferenca
0,customers,99441,99441,0
1,geolocation,1000163,1000163,0
2,orders,99441,99441,0
3,items,112650,112650,0
4,payments,103886,103886,0
5,reviews,99224,104719,5495
6,products,32951,32951,0
7,sellers,3095,3095,0
8,category_translation,71,71,0


### 1b · Investigue a tabela que não bate
Encontre um registro de `reviews` cujo texto contenha quebra de linha. O que isso diz sobre usar `wc -l` (comando do Linux que conta as linhas de um arquivo) como controle de carga, isto é, como verificação de que tudo o que saiu da fonte chegou ao destino?

In [11]:
sql("""
SELECT COUNT(*) AS reviews_com_quebra_de_linha
FROM reviews
WHERE review_comment_message LIKE '%' || chr(10) || '%'
""")

,reviews_com_quebra_de_linha
0,3852


In [12]:
display(sql("""
SELECT SUM(length(coalesce(review_comment_message, '')) - length(replace(coalesce(review_comment_message, ''), chr(10), '')))
     + SUM(length(coalesce(review_comment_title, ''))   - length(replace(coalesce(review_comment_title, ''),   chr(10), '')))
       AS total_de_quebras_de_linha
FROM reviews
"""))
sql("""
SELECT review_id, review_comment_message
FROM reviews
WHERE review_comment_message LIKE '%' || chr(10) || '%'
LIMIT 3
""")

,total_de_quebras_de_linha
0,5495.0


,review_id,review_comment_message
0,4b49719c8a200003f700d3d986ea1a19,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
1,fe3db7c069d694bab50cc43463f91608,A compra foi realizada facilmente.\r\nA entreg...
2,e233e51d11511bf30e568c76360ace52,recebi somente 1 controle Midea Split ESTILO.\...


## 2 · Grão e chaves
O **grão** responde: *uma linha desta tabela representa exatamente o quê?* Para cada tabela abaixo, proponha o grão e teste se a chave que você imagina é de fato única.

| Tabela | Chave que parece óbvia |
|---|---|
| customers | `customer_id` · e `customer_unique_id`? |
| orders | `order_id` |
| items | `order_id` + `order_item_id` |
| reviews | `review_id` |
| geolocation | `geolocation_zip_code_prefix` |

In [13]:
sql("""
SELECT 'customers.customer_id' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT customer_id) AS distintos FROM customers
UNION ALL SELECT 'customers.customer_unique_id', COUNT(*), COUNT(DISTINCT customer_unique_id) FROM customers
UNION ALL SELECT 'orders.order_id', COUNT(*), COUNT(DISTINCT order_id) FROM orders
UNION ALL SELECT 'items.(order_id, order_item_id)', COUNT(*), (SELECT COUNT(*) FROM (SELECT DISTINCT order_id, order_item_id FROM items)) FROM items
UNION ALL SELECT 'reviews.review_id', COUNT(*), COUNT(DISTINCT review_id) FROM reviews
UNION ALL SELECT 'geolocation.zip_code_prefix', COUNT(*), COUNT(DISTINCT geolocation_zip_code_prefix) FROM geolocation
""")

,chave,linhas,distintos
0,customers.customer_id,99441,99441
1,customers.customer_unique_id,99441,96096
2,orders.order_id,99441,99441
3,"items.(order_id, order_item_id)",112650,112650
4,reviews.review_id,99224,98410
5,geolocation.zip_code_prefix,1000163,19015


### 2b · Desdobramentos
1. Quantas **pessoas** (`customer_unique_id`) aparecem com mais de um `customer_id`? O que `customer_id` realmente identifica?
2. Quantos `review_id` se repetem? E quantos pedidos têm mais de uma avaliação?
3. Quantas linhas de `geolocation` são **duplicatas exatas** (todas as colunas iguais)?

In [14]:
sql("""
SELECT
  (SELECT COUNT(*) FROM (SELECT customer_unique_id FROM customers GROUP BY 1 HAVING COUNT(*) > 1)) AS pessoas_com_mais_de_um_id,
  (SELECT COUNT(*) FROM (SELECT review_id FROM reviews GROUP BY 1 HAVING COUNT(*) > 1))           AS review_ids_repetidos,
  (SELECT COUNT(*) FROM (SELECT order_id FROM reviews GROUP BY 1 HAVING COUNT(*) > 1))            AS pedidos_com_mais_de_uma_review,
  (SELECT COUNT(*) FROM geolocation) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM geolocation)) AS geo_duplicatas_exatas
""")

,pessoas_com_mais_de_um_id,review_ids_repetidos,pedidos_com_mais_de_uma_review,geo_duplicatas_exatas
0,2997,789,547,261831


## 3 · Tipos: o caso do CEP
O prefixo de CEP aparece em três tabelas: `customers`, `sellers` e `geolocation`.
1. Que **tipo** o DuckDB inferiu em cada uma? (`DESCRIBE tabela`, como no aquecimento)
2. Abra o arquivo bruto (as 3 primeiras linhas) de `customers` e de `geolocation`. Como o CEP está gravado em cada um?
3. Leia `geolocation` com **pandas** (`pd.read_csv`) e veja o tipo e os primeiros valores do CEP. O que aconteceu?
4. Quantos clientes têm prefixo de CEP com menos de 5 dígitos?

In [15]:
for t in ["customers", "sellers", "geolocation"]:
    tipo = [r for r in con.sql(f"DESCRIBE {t}").fetchall() if "zip" in r[0]][0]
    print(t, "->", tipo[0], tipo[1])
print()
for arq in ["olist_customers_dataset.csv", "olist_geolocation_dataset.csv"]:
    print(arq, open(f"{DATA}/{arq}", encoding="utf-8").readlines()[1:3])
g = pd.read_csv(f"{DATA}/olist_geolocation_dataset.csv")
print("\npandas:", g["geolocation_zip_code_prefix"].dtype, g["geolocation_zip_code_prefix"].head(2).tolist())
sql("SELECT COUNT(*) AS clientes_cep_com_menos_de_5_digitos FROM customers WHERE LENGTH(CAST(customer_zip_code_prefix AS VARCHAR)) < 5")

customers -> customer_zip_code_prefix BIGINT
sellers -> seller_zip_code_prefix BIGINT
geolocation -> geolocation_zip_code_prefix VARCHAR

olist_customers_dataset.csv ['06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP\n', '18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP\n']
olist_geolocation_dataset.csv ['"01037",-23.54562128115268,-46.63929204800168,sao paulo,SP\n', '"01046",-23.546081127035535,-46.64482029837157,sao paulo,SP\n']

pandas: int64 [1037, 1046]


,clientes_cep_com_menos_de_5_digitos
0,23995


## 4 · Integridade e completude entre tabelas
1. Quantos pedidos **não têm** itens? E pagamento? E avaliação?
2. Cruze `order_status` com a presença de `order_delivered_customer_date`. Existe pedido *entregue* sem data de entrega? Pedido *cancelado* com data de entrega?

In [16]:
display(sql("""
SELECT
  (SELECT COUNT(*) FROM orders o WHERE NOT EXISTS (SELECT 1 FROM items i    WHERE i.order_id = o.order_id)) AS sem_itens,
  (SELECT COUNT(*) FROM orders o WHERE NOT EXISTS (SELECT 1 FROM payments p WHERE p.order_id = o.order_id)) AS sem_pagamento,
  (SELECT COUNT(*) FROM orders o WHERE NOT EXISTS (SELECT 1 FROM reviews r  WHERE r.order_id = o.order_id)) AS sem_review
"""))
sql("""
SELECT order_status,
       COUNT(*) AS pedidos,
       COUNT(order_delivered_customer_date) AS com_data_entrega,
       COUNT(*) - COUNT(order_delivered_customer_date) AS sem_data_entrega
FROM orders GROUP BY 1 ORDER BY 2 DESC
""")

,sem_itens,sem_pagamento,sem_review
0,775,1,768


,order_status,pedidos,com_data_entrega,sem_data_entrega
0,delivered,96478,96470,8
1,shipped,1107,0,1107
2,canceled,625,6,619
3,unavailable,609,0,609
4,invoiced,314,0,314
5,processing,301,0,301
6,created,5,0,5
7,approved,2,0,2


## 5 · Consistência temporal
O ciclo de um pedido deveria seguir a ordem: compra → aprovação → entrega à transportadora → entrega ao cliente.
1. Qual o período coberto pelos pedidos?
2. Quantos pedidos violam a ordem esperada? Teste pelo menos: *entregue ao cliente antes de aprovado* e *entregue à transportadora antes da compra*.

In [17]:
sql("""
SELECT
  MIN(order_purchase_timestamp) AS primeira_compra,
  MAX(order_purchase_timestamp) AS ultima_compra,
  COUNT(*) FILTER (WHERE order_delivered_customer_date < order_approved_at)       AS entregue_antes_de_aprovado,
  COUNT(*) FILTER (WHERE order_delivered_carrier_date  < order_purchase_timestamp) AS transportadora_antes_da_compra
FROM orders
""")

,primeira_compra,ultima_compra,entregue_antes_de_aprovado,transportadora_antes_da_compra
0,2016-09-04 21:15:19,2018-10-17 17:30:18,61,166


## 6 · Padronização e semântica
1. Em `geolocation`, liste as grafias de cidade que contêm "paulo" no estado de SP, com contagem.
2. Quais categorias de `products` não existem na tabela de tradução? Quantos produtos estão sem categoria?
3. Leia os nomes das colunas de `products` com atenção. Algo errado?

In [18]:
display(sql("""
SELECT geolocation_city, COUNT(*) AS linhas
FROM geolocation WHERE geolocation_state = 'SP' AND geolocation_city ILIKE '%paulo%'
GROUP BY 1 ORDER BY 2 DESC
"""))
display(sql("""
SELECT DISTINCT product_category_name AS categoria_sem_traducao
FROM products p
WHERE product_category_name IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM category_translation t WHERE t.product_category_name = p.product_category_name)
"""))
display(sql("SELECT COUNT(*) AS produtos_sem_categoria FROM products WHERE product_category_name IS NULL"))
[c for c in sql("SELECT * FROM products LIMIT 0").columns]

,geolocation_city,linhas
0,sao paulo,135799
1,são paulo,24917
2,paulo de faria,56
3,sãopaulo,2
4,sa£o paulo,1


,categoria_sem_traducao
0,portateis_cozinha_e_preparadores_de_alimentos
1,pc_gamer


,produtos_sem_categoria
0,610


['product_id',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

## 7 · Desafio extra: os valores fecham?
Para cada pedido, compare a soma de `price + freight_value` dos itens com a soma de `payment_value` dos pagamentos. Quantos pedidos diferem em mais de R$ 1,00?

**Atenção:** levante hipóteses para a diferença, mas **não conclua** sem evidência. Dado bruto não explica a si mesmo; quem explica é o processo de negócio que o gerou.

In [19]:
sql("""
WITH i AS (SELECT order_id, SUM(price + freight_value) AS v_itens FROM items GROUP BY 1),
     p AS (SELECT order_id, SUM(payment_value) AS v_pago FROM payments GROUP BY 1)
SELECT COUNT(*) AS pedidos_com_diferenca_maior_que_1_real
FROM i JOIN p USING (order_id)
WHERE ABS(v_itens - v_pago) > 1
""")

,pedidos_com_diferenca_maior_que_1_real
0,312


## 8 · Do dado à pergunta: mapa de uso
Para cada pergunta de negócio, indique **quais tabelas** responde, **em que grão** a resposta sai e **qual achado de hoje** atrapalha se ninguém tratar.

| Pergunta de negócio | Tabelas | Grão da resposta | Achado que atrapalha |
|---|---|---|---|
| Faturamento mensal por categoria | `items`, `orders`, `products`, `category_translation` | mês × categoria | 610 produtos sem categoria e 2 categorias sem tradução somem do relatório; pedidos `canceled`/`unavailable` precisam de regra de inclusão; o período começa em set/2016 e termina em out/2018, com meses parciais nas pontas |
| Quantos clientes compraram mais de uma vez | `customers`, `orders` | pessoa (`customer_unique_id`) | `customer_id` é um ID por pedido: contado por ele, ninguém recompra. Por pessoa, são 2.997 clientes recorrentes |
| Prazo médio de entrega por estado | `orders`, `customers` | UF do cliente | 8 pedidos entregues sem data de entrega; 61 entregues antes de aprovados geram prazos negativos; pedidos não entregues precisam ser filtrados |
| Nota média de avaliação por vendedor | `reviews`, `orders`, `items` | vendedor | 789 avaliações aparecem em vários pedidos e 547 pedidos têm várias avaliações: o JOIN duplica notas. Pedido com mais de um vendedor atribui a mesma nota a todos. 768 pedidos sem avaliação |
| Distância média entre vendedor e cliente | `customers`, `sellers`, `geolocation`, `items` | par vendedor-cliente de cada pedido | CEP lido como número em clientes e vendedores (23.995 clientes sem o zero) quebra o JOIN; `geolocation` tem 52 linhas por prefixo e 261.831 duplicatas, então é preciso agregar uma coordenada por CEP antes |

## Tabela de Achados (entregue ao fim da prática)
Uma linha por problema. Ela será o ponto de partida das próximas aulas.

| # | Tabela(s) | Problema | Evidência (número) | Impacto se não tratar | Tratamento provável |
|---|---|---|---|---|---|
| 1 | reviews | Comentários com quebra de linha | 104.719 linhas físicas × 99.224 registros (5.495 a mais); 3.852 avaliações com quebra | Controle de carga por `wc -l` acusa perda que não existe | Contar registros após o parse, não linhas do arquivo |
| 2 | customers | `customer_id` não identifica a pessoa | 99.441 IDs para 96.096 pessoas; 2.997 com mais de um ID | Recompra fica invisível | Usar `customer_unique_id` para análises por cliente |
| 3 | reviews | `review_id` não é único | 98.410 distintos em 99.224 linhas; 789 repetidos; 547 pedidos com mais de uma avaliação | Chave errada; JOINs duplicam notas | Chave composta (`review_id`, `order_id`) |
| 4 | geolocation | Duplicatas exatas e vários pontos por prefixo | 261.831 duplicatas; 1.000.163 linhas para 19.015 prefixos | JOIN pelo CEP multiplica linhas | Remover duplicatas e agregar uma coordenada por prefixo |
| 5 | customers, sellers | CEP inferido como número | Tipo BIGINT em clientes e vendedores, VARCHAR em geolocation; 23.995 clientes com menos de 5 dígitos | JOIN com geolocation perde cerca de 24% dos clientes | Converter para texto de 5 dígitos com `lpad` |
| 6 | orders, items, payments, reviews | Pedidos incompletos | 775 sem itens; 1 sem pagamento; 768 sem avaliação | Totais por pedido divergem conforme a tabela de partida | Manter e marcar; decidir inclusão por status |
| 7 | orders | Status incoerente com a data de entrega | 8 `delivered` sem data; 6 `canceled` com data | Prazos e taxas de cancelamento errados | Flags; decisão do negócio para os cancelados |
| 8 | orders | Ordem temporal violada | 61 entregues antes da aprovação; 166 coletados antes da compra | Prazos negativos nas métricas de entrega | Marcar, sem imputar datas |
| 9 | geolocation | Grafia e codificação de cidades | `sao paulo` 135.799; `são paulo` 24.917; `sãopaulo` 2; `sa£o paulo` 1 | Agregações por cidade fragmentadas | Padronizar texto (acentos, espaços) e corrigir a codificação |
| 10 | products | Categoria ausente ou sem tradução | 610 produtos sem categoria; 2 categorias fora da tabela de tradução | Faturamento por categoria incompleto | Categoria "sem_categoria"; completar a tradução |
| 11 | products | Nomes de colunas com erro de digitação | `product_name_lenght`, `product_description_lenght` | Erros e confusão no código que consome a tabela | Renomear na camada silver |
| 12 | items, payments | Valores do pedido não fecham | 312 pedidos com diferença acima de R$ 1,00 | Conciliação financeira inconsistente | Investigar juros e vouchers antes de qualquer correção |
| 13 | payments | Pagamentos suspeitos | 3 pagamentos `not_defined` com valor 0; parcelas com mínimo 0 | Distorce análises de meio de pagamento | Marcar para revisão |